In [3]:
import os
import json
import gzip
import itertools
from itertools import chain
import pandas as pd
import numpy as np
import networkx as nx

import scipy.sparse as spsparse
import scipy.stats as spstats

import matplotlib.pylab as plt
import seaborn as sns
from tqdm import tqdm

from statsmodels.stats.multitest import multipletests as holm
import statsmodels.api as sm
#import pyscisci.all as pyscisci

from scipy.stats import pearsonr

import warnings
warnings.filterwarnings("ignore")

In [4]:
colours=['#8D95A0','#2171b5','#DA6437']

In [ ]:
fieldinfo = pd.read_csv('../data/raw/fields_data/fieldinfo0.csv.gz')
fieldinfo0 = fieldinfo[fieldinfo['FieldLevel']==1]
df_field_names = fieldinfo[fieldinfo['FieldlevelName']=='fields'][['FieldId','FieldName']]

In [88]:
fieldinfo[fieldinfo['FieldLevel']==3]

,FieldId,FieldName,FieldLevel,FieldlevelName,WikiData,NumberPublications,NumberCitations,FieldDescription
282,10077,Molecular Mechanisms of Synaptic Plasticity an...,3,topics,https://en.wikipedia.org/wiki/Synaptic_plasticity,188189,7229989,This cluster of papers explores the molecular ...
283,10002,Advancements in Density Functional Theory,3,topics,https://en.wikipedia.org/wiki/Density_function...,174372,6253608,This cluster of papers represents advancements...
284,10001,Tectonic and Geochronological Evolution of Oro...,3,topics,https://en.wikipedia.org/wiki/Geochronology,216444,5976114,This cluster of papers focuses on the tectonic...
285,10015,RNA Sequencing Data Analysis,3,topics,https://en.wikipedia.org/wiki/RNA-Seq,108711,5732472,This cluster of papers focuses on the statisti...
286,10017,Climate Change and Paleoclimatology,3,topics,https://en.wikipedia.org/wiki/Paleoclimatology,267317,4573322,This cluster of papers encompasses research on...
...,...,...,...,...,...,...,...,...
4793,13758,Regulation and Governance of Emerging Technolo...,3,topics,https://en.wikipedia.org/wiki/Technology_neutr...,425,180,This cluster of papers explores the regulation...
4794,13431,"Education, Innovation, and Organizational Deve...",3,topics,https://en.wikipedia.org/wiki/Education_and_in...,540,146,This cluster of papers covers a wide range of ...
4795,13616,Sustainable Development and Corporate Complian...,3,topics,https://en.wikipedia.org/wiki/Sustainable_deve...,177,135,This cluster of papers explores the intersecti...
4796,14087,Interdisciplinary Research in Various Fields,3,topics,https://en.wikipedia.org/wiki/Interdisciplinar...,73,20,This cluster of papers covers a wide range of ...


In [ ]:
domain_id=1

field_hierarchy=pd.read_csv('../data/raw/fields_data/fieldhierarchy0.csv.gz')
field_to_domain=field_hierarchy[field_hierarchy['ParentFieldId'].isin([1,2,3,4])]

fields=field_to_domain[field_to_domain['ParentFieldId']==1]
field_ids=fields['ChildFieldId'].to_list()


domain_dic = fieldinfo[fieldinfo['FieldLevel']==0].set_index('FieldId')['FieldName'].to_dict()

domain_name=domain_dic[domain_id]

{3: 'Physical Sciences',
 4: 'Health Sciences',
 1: 'Life Sciences',
 2: 'Social Sciences'}

In [10]:
import SelfCitation_12272025 as sc

In [46]:
import importlib

importlib.reload(sc)

<module 'SelfCitation_12272025' from '/Users/psp2nq/Documents/NationalBiasOwn/SelfCitation_12272025.py'>

In [ ]:
dfauc=pd.read_csv('../data/raw/CountryData/oa_countrycites_noselfauthoraff_CitingFieldId_auc.csv.gz')

In [13]:
dfauc

,CitingCountry,CitingFieldId,CitedYear,CitedCountry,AUC,Cov,N
0,AD,12,1985,GB,NaN,NaN,1
1,AD,27,1985,GB,NaN,NaN,1
2,AD,35,1985,GB,NaN,NaN,1
3,AD,12,1986,NaN,NaN,NaN,0
4,AD,12,1986,ES,0.5,NaN,1
...,...,...,...,...,...,...,...
9397812,ZW,36,2024,RW,0.5,NaN,1
9397813,ZW,36,2024,ZA,0.5,NaN,1
9397814,ZW,36,2024,NP,0.5,NaN,1
9397815,ZW,36,2024,IE,0.5,NaN,1


In [48]:
dflist=[]

for field in tqdm(range(11,37)):
        
    self_auc_df=dfauc[dfauc['CitingFieldId']==field]
    
    self_auc=sc.data_preprocess(self_auc_df)
    dfself_vars=sc.compile_dataset_fields(self_auc, field)
    
    income=sc.income()
    
    dfself_vars=dfself_vars.merge(income, how='left', on=['Year','Country'])
    
    dfself_vars = sc.add_polity2(dfself_vars)
    dfself_vars = sc.add_goverment_index(dfself_vars)
    
    del dfself_vars['country_code']
    del dfself_vars['date']
    
    dfself_vars = sc.regression_data_preprocess(dfself_vars, 50,1990,2019,False)
    
    dfself_vars['field_id'] = field
    
    dflist.append(dfself_vars)

dfself_fields=pd.concat(dflist)
    

  4%|█▋                                          | 1/26 [00:01<00:35,  1.40s/it]

number of countries from 1990 to 2019: 175
number of countries without nan and pub>50: 80


  8%|███▍                                        | 2/26 [00:02<00:32,  1.36s/it]

number of countries from 1990 to 2019: 141
number of countries without nan and pub>50: 53


 12%|█████                                       | 3/26 [00:04<00:31,  1.39s/it]

...

number of countries from 1990 to 2019: 127
number of countries without nan and pub>50: 25


 96%|█████████████████████████████████████████▎ | 25/26 [00:32<00:01,  1.26s/it]

number of countries from 1990 to 2019: 119
number of countries without nan and pub>50: 31


100%|███████████████████████████████████████████| 26/26 [00:33<00:00,  1.29s/it]

number of countries from 1990 to 2019: 170
number of countries without nan and pub>50: 71


In [50]:
dfself_fields.columns

Index(['CitingCountry', 'CitingFieldId', 'Year', 'Country', 'AUC', 'Cov', 'N',
       'zscore', 'pvalue', 'significant', 'NumPub', 'TopJournal', 'FracTop',
       'OANumPub', 'OATopJournal', 'OAFracTop', 'normalized_frac_top',
       'logNumPub', 'FractionNationalAuthors', 'FracInternationalAuthors',
       'TopicDiversity1', 'TopicDiversity2', 'RND_per', 'PAT_res', 'PAT_nres',
       'GDP', 'GDP_PCAP', 'GNI', 'GNI_PCAP', 'NResearchers', 'Pop',
       'PAT_total', 'logPop', 'income_level', 'polity2', 'is_democratic',
       'cap_sum_index', 'gov_sum_index', 'sig_direction', 'developed',
       'income_group', 'logzscore', 'field_id'],
      dtype='object')

In [ ]:
dfself_fields.to_csv('../data/clean/noselfauthor_fields_R_03172026.csv.gz', index=False, sep=',')

# Add disruption rate, novelty rate and hit rate

In [ ]:
df_novel=pd.read_csv('../data/clean/country_novelty_disruption_fields.csv.gz')
del df_novel['num_pubs']

dfself=pd.read_csv("../data/clean/noselfauthor_fields_R_03172026.csv.gz")
dfself.rename(columns={'field_id':'field'}, inplace=True)


dfmerge=dfself.merge(df_novel, how='left', left_on=['Country','Year','field'], right_on=['country_code','year','field'])
del dfmerge['country_code']
del dfmerge['year']

dfhit=pd.read_csv('../data/clean/hit_rate_field_per_country_year.csv.gz')

dfhit.rename(columns={'FieldId':'field'}, inplace=True)

dfmerge=dfmerge.merge(dfhit[['Country','Year','field','hit_rate']], how='left', on=['Country','Year','field'])

dfmerge.to_csv('../data/clean/noselfauthor_fields_R_disruption_03172026.csv.gz', 
                index=False, sep=',')